1. chromadb入门
1.1 Client客户端
客户端用于连接chroma向量数据库。

临时客户端：client = chromadb.Client()
持久客户端：client = chromadb.PersistentClient(path="/path/to/save/to")
HTTP客户端：client = chromadb.HttpClient(host='localhost', port=8000)
异步HTTP客户端：client = chromadb.AsyncHttpClient(host='localhost', port=8000)
1.2 collection
collection，集合，相当于mysql当中的database/table的概念,一个chroma当中可以有多个collection。

创建collection: client.create_collection(name="my_collection", embedding_function=emb_fn)
获取collection: client.get_collection(name="my_collection", embedding_function=emb_fn)
获取/创建collection: client.get_or_create_collection(name="my_collection", embedding_function=emb_fn)
删除collection: client.delete_collection(name="my_collection")
collection的实例方法：

添加: add(ids=[], documents=[], embeddings=[]) -> None
语义查询: query(query_texts=[], n_results=10) -> QueryResult
查询: get(ids=[]) -> GetResult
查询文档数量: count() -> Number

1.定义embedding模型

In [ ]:
import os
from openai import OpenAI
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv

load_dotenv()

from langchain_community.embeddings import DashScopeEmbeddings
embeddings=DashScopeEmbeddings(model='text-embedding-v4',
dashscope_api_key=os.environ['DASHSCOPE_API_KEY'])



2.读入文档并切割

In [ ]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader('/root/ai_rag_project/data_base/科技行业 2025 年展望.txt')
docs = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=100)
all_splitters = text_splitter.split_documents(docs)


3.存储到向量数据库

In [ ]:
from langchain_chroma import Chroma
vector_store = Chroma(embedding_function=embeddings)
ids = vector_store.add_documents(all_splitters)


4.检索生成

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {query} 
Context: {context} 
Answer:""")

llm=ChatOpenAI(api_key=os.environ['DASHSCOPE_API_KEY'],
               base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
               model="qwen3.6-plus"
               )

def query_vector(info):
    retriever=vector_store.as_retriever()
    docs = retriever.invoke(info["query"])
    docs_str = '\n\n'.join(doc.page_content for doc in docs)
    return docs_str

output_parser = StrOutputParser()

rag_chain = ( {"context":query_vector, "query": lambda x: x["query"]}| prompt | llm | output_parser)

result = rag_chain.invoke({'query':"2025年AI服务器出货量预计是多少"})





a.根据问题到向量数据库中进行检索
b.将检索到的文档片段加到prompt中，并进行生成

In [ ]:
# retriever=vector_store.as_retriever
# def query_vector(info):
#     retriever=vector_store.as_retriever()
#     docs = retriever.invoke(info["query"])
#     docs_str = '\n\n.join(doc.page_content for doc in docs)'
#     return docs_str

# query_vector({'query':"2025年AI服务器出货量预计是多少"})



langchaing中的检索器和压缩器
1.1 检索器 Retriever
检索器，就是根据给定的问题，检索之后，返回对应的文档列表。langchain对检索器封装的基类为BaseRetriever。

检索器列表：

VectorStoreRetriever：向量存储检索器。（Chroma实例方法as_retriever()返回的就是这个检索器）
EnsembleRetriever: 组合多个检索器的检索器（混合检索时使用）
MultiQueryRetriever: 使用LLM将原始问题进行改写成多个问题，再进行检索！
TimeWeightedVectorStoreRetriever: 结合了相似度和新鲜度的向量存储检索器。
。。。。。。
1.2 压缩器 Compressor
压缩器，用于将检索器检索到的文档列表进行后处理。langchain对压缩器封装的基类为BaseDocumentCompressor，所以压缩器更准确的称呼为“文档压缩器”。

压缩器列表

CrossEncoderReranker: 使用CrossEncoder(交叉编码器)对文档列表进行重排！
LLMChainExtractor: 根据问题从召回文档列表中提取关键句。
LLMChainFilter: 用于判断召回文本是否与问题相关。
DocumentCompressorPipeline: 用管道的方式串联2个或多个压缩器。
。。。。。。
CrossEncoderReranker

参数：

model: 模型对象。
top_n: int,返回几个文档列表
from langchain.retrievers.document_compressors import CrossEncoderReranker
1.3 检索+压缩器整合
ContextualCompressionRetriever，将检索器和压缩器串联起来，实现检索和压缩的效果的检索器。

参数：

base_retriever: (必填), 基础检索器
base_compressor: (必填),基础压缩器
方法：

invoke(): 同步调用
ainvoke(): 异步调用
stream(): 同步流式调用
astream(): 异步流式调用
from langchain.retrievers import ContextualCompressionRetriever
2. LangChain对CrossEncoder的封装
实际上底层也是使用sentence_transformers来加载CrossEncoder模型。

from langchain_community.cross_encoders import HuggingFaceCrossEncoder
参数：

model_name: 模型名称。如：BAAI/bge-reranker-large
方法：

score(): 根据传入的文档列表打分。